In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q torch transformers accelerate safetensors pillow pandas numpy matplotlib scikit-learn scipy seaborn

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
from pathlib import Path

# Explore your Google Drive structure
drive_path = Path('/content/drive/MyDrive')

print("📁 Google Drive Structure:\n")
for item in drive_path.iterdir():
    if item.is_dir():
        print(f"📂 {item.name}/")
        try:
            for subitem in list(item.iterdir())[:5]:  # Show first 5 items
                if subitem.is_dir():
                    print(f"   📂 {subitem.name}/")
                else:
                    print(f"   📄 {subitem.name}")
            remaining = len(list(item.iterdir())) - 5
            if remaining > 0:
                print(f"   ... and {remaining} more items")
        except PermissionError:
            print(f"   (Permission denied)")

📁 Google Drive Structure:

📂 suresh pics/
   📄 IMG_0077.heic
   📄 IMG_0076.heic
   📄 IMG_0078.heic
   📄 IMG_0081.heic
   📄 IMG_0080.heic
   ... and 62 more items
📂 videos/
   📄 IMG_0059.HEIC
   📄 IMG_0060.HEIC
   📄 IMG_0172.HEIC
   📄 IMG_0184.HEIC
   📄 IMG_0221.MOV
   ... and 12 more items
📂 Travel buddy /
   📄 IMG_1598.HEIC
   📄 IMG_1588.HEIC
   📄 IMG_1318.HEIC
   📄 IMG_1325.HEIC
   📄 IMG_1493 (1).HEIC
   ... and 316 more items
📂 Colab Notebooks/
   📄 EDA_ship_routes.ipynb
   📄 Copy of TACC_SC26.ipynb
📂 feldmochinger see/
   📄 IMG_4242.MOV
   📄 IMG_4246.HEIC
   📄 IMG_4250.HEIC
   📄 IMG_4224.HEIC
   📄 IMG_4228.HEIC
   ... and 12 more items
📂 Adventures /
   📄 03d64225-cda3-4072-8e38-aa447cc9e792.JPG
   📄 IMG_0887.HEIC
   📄 IMG_1953.HEIC
   📄 IMG_9744.JPG
   📄 GOPR6702.JPG
   ... and 4 more items
📂 CVPR JSON/
   📄 detection_results copy.json
   📄 detection_results copy 2_part1.json
   📄 detection_results_part2.json
📂 GENA I/
   📂 Automation/
📂 TACC EXPERIMENTS/
   📂 205_Post_Def_rgb/
  

In [3]:
import os

data_dir = '/content/drive/MyDrive/TACC EXPERIMENTS'
if os.path.exists(data_dir):
    print(f"Contents of {data_dir}:")
    for item in os.listdir(data_dir):
        item_path = os.path.join(data_dir, item)
        if os.path.isdir(item_path):
            print(f"[DIR]  {item}")
        else:
            print(f"[FILE] {item}")
else:
    print(f"Directory {data_dir} does not exist.")

Contents of /content/drive/MyDrive/TACC EXPERIMENTS:
[DIR]  205_Post_Def_rgb
[DIR]  part 2_pre_def_rgb
[DIR]  Part_one_pre_def_rgb
[DIR]  Post_def_rgb_part1
[DIR]  part3_post_def_rgb
[DIR]  part4_post_def_rgb
[DIR]  .ipynb_checkpoints


In [4]:
!find "/content/drive/MyDrive/TACC EXPERIMENTS" -type f -name "._*" -delete

In [5]:
!find "/content/drive/MyDrive/TACC EXPERIMENTS" -type f -name "._*" | head

In [6]:
%%writefile advanced_figures_cvpr_neurips.py
import argparse
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageFile
from sklearn.decomposition import PCA
import scipy.ndimage as ndimage

ImageFile.LOAD_TRUNCATED_IMAGES = True
VALID_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}

def is_valid_image_file(f: Path) -> bool:
    return (
        f.is_file()
        and f.suffix.lower() in VALID_EXTS
        and not f.name.startswith("._")
        and not f.name.startswith(".")
    )

def classify_stage(name: str) -> str:
    low = name.lower()
    if "pre" in low:
        return "pre_defoliation"
    if "post" in low:
        return "post_defoliation"
    if "rose" in low:
        return "rose_nursery"
    return "unknown"

def compute_white_boll_proxy(img: Image.Image, stage: str = "unknown"):
    arr = np.array(img.convert("RGB")).astype(np.float32)
    r, g, b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]

    intensity = (r + g + b) / 3.0
    max_c = np.max(arr, axis=2)
    min_c = np.min(arr, axis=2)
    saturation = (max_c - min_c) / (max_c + 1e-6)

    # Use Excess Green (ExG) to explicitly reject green vegetation
    exg = 2 * g - r - b

    if stage == "post_defoliation":
        # Stricter thresholds for post-defoliation to reduce over-counting
        mask = (intensity > 160) & (saturation < 0.15) & (exg < 5) & (r > 140) & (b > 140)
        # Larger opening to remove more false positive fragments
        mask = ndimage.binary_opening(mask, structure=np.ones((5, 5)))
    else:
        # PRE-DEFOLIATION LOGIC (Untouched as requested)
        mask = (intensity > 150) & (saturation < 0.20) & (exg < 10) & (r > 130) & (b > 130)
        mask = ndimage.binary_opening(mask, structure=np.ones((4, 4)))

    # Generate a gaussian heatmap of the bolls
    heatmap = ndimage.gaussian_filter(mask.astype(float), sigma=25)

    # Connected components
    labeled_mask, num_features = ndimage.label(mask)
    slices = ndimage.find_objects(labeled_mask)

    bboxes = []
    for sl in slices:
        if sl is None:
            continue
        y_slice, x_slice = sl
        h = y_slice.stop - y_slice.start
        w = x_slice.stop - x_slice.start

        # Restrict size limits so we only grab legit bolls
        if 5 <= w <= 150 and 5 <= h <= 150:
            area = w * h
            if 25 <= area <= 6000:
                aspect = w / float(h)
                if 0.2 <= aspect <= 5.0:
                    bboxes.append((x_slice.start, y_slice.start, w, h))

    proxy_count = len(bboxes)
    proxy_area = float(mask.mean())
    return bboxes, proxy_count, proxy_area, heatmap

def extract_features(img: Image.Image, stage: str = "unknown"):
    arr = np.array(img.convert("RGB")).astype(np.float32)
    r, g, b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]
    exg = 2 * g - r - b
    ngrdi = (g - r) / (g + r + 1e-6)
    rbr = r / (b + 1e-6)
    brightness = (r + g + b) / 3.0

    _, proxy_count, proxy_area, _ = compute_white_boll_proxy(img, stage)

    return {
        "mean_r": float(r.mean()),
        "mean_g": float(g.mean()),
        "mean_b": float(b.mean()),
        "mean_exg": float(exg.mean()),
        "std_exg": float(exg.std()),
        "mean_ngrdi": float(ngrdi.mean()),
        "mean_rbr": float(rbr.mean()),
        "bright_fraction": float((brightness > 180).mean()),
        "boll_count_proxy": float(proxy_count),
        "white_region_fraction": float(proxy_area),
    }

def build_table(root_dir: Path, sample_limit=80):
    rows = []
    exemplar_paths = {"pre_defoliation": [], "post_defoliation": []}

    for directory in sorted(root_dir.rglob("*")):
        if not directory.is_dir():
            continue
        files = sorted([f for f in directory.iterdir() if is_valid_image_file(f)])
        if not files:
            continue

        stage = classify_stage(directory.name)
        keep = files[:sample_limit]

        for f in keep:
            try:
                t0 = time.time()
                img = Image.open(f).convert("RGB")
                feats = extract_features(img, stage)
                elapsed = time.time() - t0
                rows.append({
                    "directory": directory.name,
                    "stage": stage,
                    "path": str(f),
                    "latency_s": elapsed,
                    **feats,
                })
                if stage in exemplar_paths and len(exemplar_paths[stage]) < 6:
                    exemplar_paths[stage].append(str(f))
            except Exception:
                pass

    return pd.DataFrame(rows), exemplar_paths

def add_clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def make_pca_publication(df: pd.DataFrame, out_dir: Path):
    feat_cols = [
        "mean_r", "mean_g", "mean_b",
        "mean_exg", "std_exg", "mean_ngrdi", "mean_rbr",
        "bright_fraction", "boll_count_proxy", "white_region_fraction"
    ]
    plot_df = df[df["stage"].isin(["pre_defoliation", "post_defoliation"])].copy()
    if plot_df.empty:
        return
    X = plot_df[feat_cols].fillna(0.0).values
    pca = PCA(n_components=2, random_state=42)
    Z = pca.fit_transform(X)
    plot_df["pc1"] = Z[:, 0]
    plot_df["pc2"] = Z[:, 1]

    fig, ax = plt.subplots(figsize=(7.2, 5.6), dpi=300)
    for stage, marker in [("pre_defoliation", "o"), ("post_defoliation", "^")]:
        sub = plot_df[plot_df["stage"] == stage]
        ax.scatter(sub["pc1"], sub["pc2"], s=34, alpha=0.82, marker=marker, label=stage.replace("_", "-"))

    add_clean_axes(ax)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
    ax.set_title("Stage separation in image-level agricultural descriptor space", pad=10)
    ax.legend(frameon=False, loc="best")
    ax.grid(alpha=0.16, linewidth=0.6)
    plt.tight_layout()
    plt.savefig(out_dir / "fig_pca_publication.png", bbox_inches="tight")
    plt.close(fig)

def make_six_panel_grid(exemplar_paths, out_dir: Path):
    pre_paths = exemplar_paths.get("pre_defoliation", [])
    post_paths = exemplar_paths.get("post_defoliation", [])

    if not pre_paths or not post_paths:
        print("Not enough images to make the 6-panel grid.")
        return

    pre_img_path = pre_paths[0]
    post_img_path = post_paths[0]

    pre_img = Image.open(pre_img_path).convert("RGB")
    post_img = Image.open(post_img_path).convert("RGB")

    pre_bboxes, pre_count, pre_frac, pre_hm = compute_white_boll_proxy(pre_img, "pre_defoliation")
    post_bboxes, post_count, post_frac, post_hm = compute_white_boll_proxy(post_img, "post_defoliation")

    fig, axes = plt.subplots(2, 3, figsize=(18, 10), dpi=300)

    # ROW 1: Pre-Defoliation
    axes[0, 0].imshow(np.array(pre_img))
    axes[0, 0].set_title("Pre-Defoliation: Raw Image", fontsize=14, fontweight='bold')
    axes[0, 0].axis("off")

    axes[0, 1].imshow(pre_hm, cmap='hot')
    axes[0, 1].set_title("Pre-Defoliation: Heatmap", fontsize=14, fontweight='bold')
    axes[0, 1].axis("off")

    axes[0, 2].imshow(np.array(pre_img))
    for bbox in pre_bboxes:
        rect = patches.Rectangle((bbox[0], bbox[1]), bbox[2], bbox[3], linewidth=1.5, edgecolor='lime', facecolor='none')
        axes[0, 2].add_patch(rect)
    axes[0, 2].set_title(f"Pre-Defoliation Boll Count: {pre_count}", fontsize=14, fontweight='bold', color='green')
    axes[0, 2].axis("off")

    # ROW 2: Post-Defoliation
    axes[1, 0].imshow(np.array(post_img))
    axes[1, 0].set_title("Post-Defoliation: Raw Image", fontsize=14, fontweight='bold')
    axes[1, 0].axis("off")

    axes[1, 1].imshow(post_hm, cmap='hot')
    axes[1, 1].set_title("Post-Defoliation: Heatmap", fontsize=14, fontweight='bold')
    axes[1, 1].axis("off")

    axes[1, 2].imshow(np.array(post_img))
    for bbox in post_bboxes:
        rect = patches.Rectangle((bbox[0], bbox[1]), bbox[2], bbox[3], linewidth=1.5, edgecolor='lime', facecolor='none')
        axes[1, 2].add_patch(rect)
    axes[1, 2].set_title(f"Post-Defoliation Boll Count: {post_count}", fontsize=14, fontweight='bold', color='green')
    axes[1, 2].axis("off")

    fig.suptitle("Cotton Defoliation Stage Comparison: Raw vs. Heatmap vs. Detection", fontsize=18, y=1.02)
    plt.tight_layout()
    plt.savefig(out_dir / "fig_2x3_pre_post_grid.png", bbox_inches="tight")
    plt.close(fig)

def make_quantitative_panel(df: pd.DataFrame, out_dir: Path):
    plot_df = df[df["stage"].isin(["pre_defoliation", "post_defoliation"])].copy()
    if plot_df.empty:
        return
    summary = (
        plot_df.groupby("stage", as_index=False)
        .agg(
            samples=("path", "count"),
            avg_latency_s=("latency_s", "mean"),
            mean_boll_count_proxy=("boll_count_proxy", "mean"),
            mean_white_region_fraction=("white_region_fraction", "mean"),
            mean_exg=("mean_exg", "mean"),
        )
    )
    summary.to_csv(out_dir / "stage_summary_table.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.2), dpi=300)

    for stage in ["pre_defoliation", "post_defoliation"]:
        sub = plot_df[plot_df["stage"] == stage]
        axes[0].scatter(sub["white_region_fraction"], sub["boll_count_proxy"], s=24, alpha=0.78, label=stage.replace("_", "-"))
    add_clean_axes(axes[0])
    axes[0].set_xlabel("White-region fraction")
    axes[0].set_ylabel("Cotton visibility proxy (Boll Count)")
    axes[0].set_title("Visibility density vs proxy count")
    axes[0].legend(frameon=False)
    axes[0].grid(alpha=0.16, linewidth=0.6)

    x = np.arange(len(summary))
    width = 0.36
    axes[1].bar(x - width/2, summary["mean_boll_count_proxy"], width=width, label="Mean proxy count")
    axes[1].bar(x + width/2, summary["mean_white_region_fraction"] * 100, width=width, label="Mean white fraction (x100)")
    add_clean_axes(axes[1])
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(summary["stage"].str.replace("_", "-"))
    axes[1].set_title("Stage-wise contrast in cotton visibility")
    axes[1].legend(frameon=False)
    axes[1].grid(axis="y", alpha=0.16, linewidth=0.6)

    plt.tight_layout()
    plt.savefig(out_dir / "fig_quantitative_panel.png", bbox_inches="tight")
    plt.close(fig)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--root-dir", required=True)
    parser.add_argument("--out-dir", required=True)
    parser.add_argument("--sample-limit", type=int, default=80)
    parser.add_argument("--per-stage", type=int, default=3)
    args = parser.parse_args()

    root_dir = Path(args.root_dir)
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    df, exemplar_paths = build_table(root_dir, sample_limit=args.sample_limit)
    df.to_csv(out_dir / "inventory_analysis_table.csv", index=False)

    make_pca_publication(df, out_dir)
    make_six_panel_grid(exemplar_paths, out_dir)
    make_quantitative_panel(df, out_dir)

    print("Saved figures to:", out_dir)

if __name__ == "__main__":
    main()


Overwriting advanced_figures_cvpr_neurips.py


In [7]:
import gc
import torch
import pandas as pd
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from pathlib import Path

def compare_vlms(image_paths, model_configs, max_samples=4):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    all_records = []

    prompt_text = """
    Analyze this UAV agricultural image of a cotton field.
    Identify and provide a concise agronomic interpretation on:
    1. Crop stage (pre-defoliation vs post-defoliation).
    2. Visible cotton structures and an estimation of yield potential based on boll density.
    3. How this visual data and defoliation status can be used for inventory management and harvest logistics in precision agriculture.
    """

    for model_name, model_id in model_configs.items():
        print(f"\n{'='*50}")
        print(f"🔄 Loading {model_name} from: {model_id}")
        print(f"{'='*50}")

        try:
            # Clear cache before loading
            torch.cuda.empty_cache()
            gc.collect()

            processor = AutoProcessor.from_pretrained(model_id)
            model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
                model_id,
                torch_dtype=torch.float16 if device=="cuda" else torch.float32,
                device_map="auto"
            )

            for p in image_paths[:max_samples]:
                img = Image.open(p).convert("RGB")
                # Downsize high-res UAV images to a safe multiple (e.g., 784) to prevent CUDA device-side asserts
                img.thumbnail((784, 784), Image.Resampling.LANCZOS)

                messages = [
                    {
                        "role": "user",
                        "content": [
                            {"type": "image", "image": img},
                            {"type": "text", "text": prompt_text}
                        ]
                    }
                ]

                text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                inputs = processor(text=[text], images=[img], return_tensors="pt", padding=True)

                # Safely move dictionary values to the correct device
                inputs = {k: v.to(device) for k, v in inputs.items()}

                with torch.no_grad():
                    out = model.generate(**inputs, max_new_tokens=150)

                generated_ids_trimmed = [
                    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs["input_ids"], out)
                ]
                output_text = processor.batch_decode(
                    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
                )[0]

                all_records.append({
                    "image": Path(p).name,
                    "model": model_name,
                    "llm_interpretation": output_text
                })

            # CLEAR GPU MEMORY BEFORE LOADING NEXT MODEL
            del model
            del processor
            gc.collect()
            torch.cuda.empty_cache()
            print(f"✅ Finished {model_name} and cleared GPU memory.")

        except Exception as e:
            print(f"❌ Error loading or running {model_name}: {e}")

    return pd.DataFrame(all_records)

# --- RUN THE COMPARISON ---

# Define the models you want to compare.
MODELS_TO_COMPARE = {
    "Qwen-2.5-VL": "Qwen/Qwen2.5-VL-3B-Instruct"
    # "AgroGPT": "path/to/your/agrogpt/model" # <--- UNCOMMENT AND UPDATE THIS PATH
}

data_dir = Path('/content/drive/MyDrive/TACC EXPERIMENTS')
valid_exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}
sample_images = [f for f in data_dir.rglob("*") if f.is_file() and f.suffix.lower() in valid_exts and not f.name.startswith("._")]

if sample_images:
    df_comparison = compare_vlms(sample_images, MODELS_TO_COMPARE, max_samples=4)

    # Display side by side if possible
    display(df_comparison)

    # Save results
    out_path = Path('/content/drive/MyDrive/agrogpt_results/model_comparison_results.csv')
    out_path.parent.mkdir(parents=True, exist_ok=True) # Automatically create the directory if it doesn't exist
    df_comparison.to_csv(out_path, index=False)
    print(f"\n💾 Saved comparison to {out_path}")
else:
    print("No images found.")


🔄 Loading Qwen-2.5-VL from: Qwen/Qwen2.5-VL-3B-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

✅ Finished Qwen-2.5-VL and cleared GPU memory.


,image,model,llm_interpretation
0,DJI_20250929125134_0319_D.JPG,Qwen-2.5-VL,### 1. Crop Stage\n\n**Crop Stage:** The image...
1,DJI_20250929125054_0301_D.JPG,Qwen-2.5-VL,### 1. Crop Stage (Pre-defoliation vs Post-def...
2,DJI_20250929125138_0321_D.JPG,Qwen-2.5-VL,### 1. Crop Stage\n\n**Crop Stage:** The image...
3,DJI_20250929125142_0323_D.JPG,Qwen-2.5-VL,### 1. Crop Stage\n\n**Crop Stage:** The image...



💾 Saved comparison to /content/drive/MyDrive/agrogpt_results/model_comparison_results.csv


In [8]:
!python advanced_figures_cvpr_neurips.py \
  --root-dir "/content/drive/MyDrive/TACC EXPERIMENTS" \
  --out-dir "/content/drive/MyDrive/agrogpt_pub_figs_updated_finally" \
  --sample-limit 80 \
  --per-stage 3

Saved figures to: /content/drive/MyDrive/agrogpt_pub_figs_updated_finally


In [10]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
import torch
from PIL import Image
import pandas as pd
from pathlib import Path
import json

def llm_visual_inspection_dual(
    image_paths,
    qwen_model_id="Qwen/Qwen2.5-VL-3B-Instruct",
    agrogpt_model_id=None,   # replace with your AgroGPT checkpoint path if available
    max_samples=6,
    output_csv="llm_visual_inspection_resultsss.csv",
    output_json="llm_visual_inspection_resultsss.json",
    extra_metadata=None
):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # ---------- Qwen ----------
    qwen_processor = AutoProcessor.from_pretrained(qwen_model_id)
    qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        qwen_model_id,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32
    ).to(device)
    qwen_model.eval()

    # ---------- AgroGPT ----------
    agrogpt_processor = None
    agrogpt_model = None
    if agrogpt_model_id is not None:
        agrogpt_processor = AutoProcessor.from_pretrained(agrogpt_model_id)
        agrogpt_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            agrogpt_model_id,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32
        ).to(device)
        agrogpt_model.eval()

    records = []

    for i, p in enumerate(image_paths[:max_samples]):
        p = str(p)
        img = Image.open(p).convert("RGB")

        meta = {}
        if extra_metadata is not None and p in extra_metadata:
            meta = extra_metadata[p]

        # ---------------- QWEN PROMPT ----------------
        qwen_prompt = f"""
        Analyze this UAV agricultural image of a cotton field.

        Identify:
        1. Crop stage (pre-defoliation vs post-defoliation).
        2. Visible cotton structures.
        3. Approximate harvest readiness.
        4. A short agronomic interpretation for field inspection.

        If available, consider these auxiliary signals:
        - boll_count_proxy: {meta.get('boll_count_proxy', 'NA')}
        - white_region_fraction: {meta.get('white_region_fraction', 'NA')}
        """

        qwen_messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": qwen_prompt}
                ]
            }
        ]

        qwen_text = qwen_processor.apply_chat_template(
            qwen_messages,
            tokenize=False,
            add_generation_prompt=True
        )

        qwen_inputs = qwen_processor(
            text=[qwen_text],
            images=[img],
            return_tensors="pt",
            padding=True
        )
        qwen_inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in qwen_inputs.items()}

        with torch.no_grad():
            qwen_out = qwen_model.generate(**qwen_inputs, max_new_tokens=150)

        qwen_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(qwen_inputs["input_ids"], qwen_out)
        ]
        qwen_output_text = qwen_processor.batch_decode(
            qwen_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )[0]

        # ---------------- AGROGPT PROMPT ----------------
        agrogpt_output_text = None
        if agrogpt_model is not None:
            agrogpt_prompt = f"""
            You are AgroGPT, an agronomic reasoning assistant.

            Based on this cotton UAV image and the extracted inspection signals below,
            provide:
            1. A concise inventory-management recommendation.
            2. A harvest logistics recommendation.
            3. A note on expected cotton visibility / picking readiness.

            Auxiliary signals:
            - crop_stage_hint: {meta.get('stage', 'NA')}
            - boll_count_proxy: {meta.get('boll_count_proxy', 'NA')}
            - white_region_fraction: {meta.get('white_region_fraction', 'NA')}
            - directory: {meta.get('directory', 'NA')}
            """

            agrogpt_messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": img},
                        {"type": "text", "text": agrogpt_prompt}
                    ]
                }
            ]

            agrogpt_text = agrogpt_processor.apply_chat_template(
                agrogpt_messages,
                tokenize=False,
                add_generation_prompt=True
            )

            agrogpt_inputs = agrogpt_processor(
                text=[agrogpt_text],
                images=[img],
                return_tensors="pt",
                padding=True
            )
            agrogpt_inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in agrogpt_inputs.items()}

            with torch.no_grad():
                agrogpt_out = agrogpt_model.generate(**agrogpt_inputs, max_new_tokens=150)

            agrogpt_trimmed = [
                out_ids[len(in_ids):] for in_ids, out_ids in zip(agrogpt_inputs["input_ids"], agrogpt_out)
            ]
            agrogpt_output_text = agrogpt_processor.batch_decode(
                agrogpt_trimmed,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False
            )[0]

        records.append({
            "image": p,
            "stage": meta.get("stage"),
            "directory": meta.get("directory"),
            "boll_count_proxy": meta.get("boll_count_proxy"),
            "white_region_fraction": meta.get("white_region_fraction"),
            "qwen_interpretation": qwen_output_text,
            "agrogpt_recommendation": agrogpt_output_text,
        })

    df = pd.DataFrame(records)
    df.to_csv(output_csv, index=False)

    with open(output_json, "w") as f:
        json.dump(records, f, indent=2)

    return df

In [11]:
from pathlib import Path
import os

# 1. Gather a few sample images from your dataset
data_dir = Path('/content/drive/MyDrive/TACC EXPERIMENTS')
valid_exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}
sample_images = [f for f in data_dir.rglob("*") if f.is_file() and f.suffix.lower() in valid_exts and not f.name.startswith("._")]

if sample_images:
    print(f"Found {len(sample_images)} images. Running AgroGPT (Qwen) on the first 4 images...")

    # 2. Run the function
    df_llm_results = llm_visual_inspection(sample_images, max_samples=4)

    # 3. Save the output to your drive
    output_dir = Path('/content/drive/MyDrive/agrogpt_resultsss')
    output_dir.mkdir(parents=True, exist_ok=True)
    save_path = output_dir / 'llm_inventory_management_resultsss.csv'

    df_llm_results.to_csv(save_path, index=False)
    print(f"\n✅ Successfully saved LLM interpretations to: {save_path}\n")

    # Display the results in the notebook
    display(df_llm_results)
else:
    print("No valid images found to process.")

Found 1548 images. Running AgroGPT (Qwen) on the first 4 images...


NameError: name 'llm_visual_inspection' is not defined

In [ ]:
from IPython.display import Image, display
# Run the cell above (!python advanced_figures_cvpr_neurips.py ...) to generate this plot first!
image_path = "/content/drive/MyDrive/agrogpt_pub_figs_updated/fig_2x2_pre_post_grid.png"
display(Image(image_path))

In [ ]:
%%writefile cotton_boll_ml_pac_qwen.py
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans

import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

ImageFile.LOAD_TRUNCATED_IMAGES = True

VALID_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}

def is_valid_image_file(f: Path) -> bool:
    return (
        f.is_file()
        and f.suffix.lower() in VALID_EXTS
        and not f.name.startswith("._")
        and not f.name.startswith(".")
    )

def classify_stage(name: str) -> str:
    low = name.lower()
    if "pre" in low:
        return "pre_defoliation"
    if "post" in low:
        return "post_defoliation"
    if "rose" in low:
        return "rose_nursery"
    return "unknown"

def compute_white_boll_proxy(img: Image.Image):
    arr = np.array(img.convert("RGB")).astype(np.float32)
    r, g, b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]

    brightness = (r + g + b) / 3.0
    whiteness = 255.0 - (np.abs(r - g) + np.abs(r - b) + np.abs(g - b)) / 3.0

    mask = (brightness > 185) & (whiteness > 220)

    row_hits = mask.sum(axis=1)
    col_hits = mask.sum(axis=0)

    row_clusters = np.sum((row_hits[1:] > 0) & (row_hits[:-1] == 0))
    col_clusters = np.sum((col_hits[1:] > 0) & (col_hits[:-1] == 0))

    proxy_count = int((row_clusters + col_clusters) / 2)
    proxy_area = float(mask.mean())
    return proxy_count, proxy_area

def extract_handcrafted_features(img: Image.Image):
    arr = np.array(img.convert("RGB")).astype(np.float32)
    r, g, b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]

    exg = 2 * g - r - b
    ngrdi = (g - r) / (g + r + 1e-6)
    rbr = r / (b + 1e-6)

    brightness = (r + g + b) / 3.0
    green_fraction = float((g > r).mean())
    bright_fraction = float((brightness > 180).mean())

    return {
        "mean_r": float(r.mean()),
        "mean_g": float(g.mean()),
        "mean_b": float(b.mean()),
        "mean_exg": float(exg.mean()),
        "std_exg": float(exg.std()),
        "mean_ngrdi": float(ngrdi.mean()),
        "mean_rbr": float(rbr.mean()),
        "green_fraction": green_fraction,
        "bright_fraction": bright_fraction,
    }

class QwenInspector:
    def __init__(self, model_id="Qwen/Qwen2.5-VL-3B-Instruct", enabled=True):
        self.enabled = enabled and torch.cuda.is_available()
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model_id = model_id
        self.processor = None
        self.model = None

    def load(self):
        if not self.enabled:
            return
        self.processor = AutoProcessor.from_pretrained(self.model_id, use_fast=True)
        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            self.model_id,
            torch_dtype=torch.float16,
            device_map="auto",
            low_cpu_mem_usage=True,
        )
        self.model.eval()

    def embed(self, img: Image.Image):
        if not self.enabled:
            return None
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": "Inspect this agricultural UAV image."}
            ]
        }]
        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.processor(
            text=[text],
            images=[img],
            return_tensors="pt",
            padding=True
        )
        inputs = {k: v.to(self.device) if hasattr(v, "to") else v for k, v in inputs.items()}
        with torch.no_grad():
            outputs = self.model(**inputs, output_hidden_states=True, return_dict=True)
        hidden = outputs.hidden_states[-1]
        emb = hidden.mean(dim=1).detach().float().cpu().numpy()[0]
        return emb

def build_dataset(root_dir, limit_per_dir=40, use_qwen=True):
    rows = []
    qwen = QwenInspector(enabled=use_qwen)
    qwen.load()

    root = Path(root_dir)
    for directory in sorted(root.rglob("*")):
        if not directory.is_dir():
            continue

        files = sorted([f for f in directory.iterdir() if is_valid_image_file(f)])
        if not files:
            continue

        stage = classify_stage(directory.name)
        files = files[:limit_per_dir]

        for f in files:
            try:
                t0 = time.time()
                img = Image.open(f).convert("RGB")
                proxy_count, proxy_area = compute_white_boll_proxy(img)
                feats = extract_handcrafted_features(img)
                emb = qwen.embed(img) if use_qwen else None
                latency = time.time() - t0

                row = {
                    "directory": directory.name,
                    "stage": stage,
                    "path": str(f),
                    "boll_count_proxy": proxy_count,
                    "white_region_fraction": proxy_area,
                    "latency_s": latency,
                    **feats
                }

                if emb is not None:
                    for i, v in enumerate(emb[:32]):  # keep first 32 dims only for lightweight analysis
                        row[f"qwen_{i}"] = float(v)

                rows.append(row)
            except Exception:
                pass

    return pd.DataFrame(rows)

def fit_ml_boll_estimator(df):
    feature_cols = [
        "mean_r", "mean_g", "mean_b",
        "mean_exg", "std_exg", "mean_ngrdi", "mean_rbr",
        "green_fraction", "bright_fraction", "white_region_fraction"
    ]
    X = df[feature_cols].fillna(0.0).values
    y = df["boll_count_proxy"].values

    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=8,
        random_state=42
    )
    model.fit(X, y)
    pred = model.predict(X)
    df = df.copy()
    df["boll_count_ml"] = pred
    return df, model, feature_cols

def compute_pac_effective_sample_size(df):
    counts = df.groupby("stage").size()
    counts = counts[counts > 0]
    total = counts.sum()
    alpha = np.ones(len(counts)) / len(counts)
    m = counts.values.astype(float)
    m_eff = 1.0 / np.sum((alpha ** 2) / m)
    return counts.reset_index(name="samples"), float(total), float(m_eff)

def make_outputs(df, outdir):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    stage_counts, raw_n, eff_n = compute_pac_effective_sample_size(df)

    stage_summary = (
        df.groupby("stage", as_index=False)
        .agg(
            samples=("path", "count"),
            avg_latency_s=("latency_s", "mean"),
            throughput_img_per_s=("latency_s", lambda x: 1.0 / np.mean(x) if len(x) else 0.0),
            mean_boll_count_proxy=("boll_count_proxy", "mean"),
            mean_boll_count_ml=("boll_count_ml", "mean"),
            mean_white_region_fraction=("white_region_fraction", "mean"),
            mean_exg=("mean_exg", "mean"),
            mean_ngrdi=("mean_ngrdi", "mean"),
        )
    )

    directory_summary = (
        df.groupby(["directory", "stage"], as_index=False)
        .agg(
            samples=("path", "count"),
            avg_latency_s=("latency_s", "mean"),
            mean_boll_count_proxy=("boll_count_proxy", "mean"),
            mean_boll_count_ml=("boll_count_ml", "mean"),
            mean_white_region_fraction=("white_region_fraction", "mean"),
        )
    )

    pac_table = pd.DataFrame({
        "metric": ["raw_sample_size", "effective_sample_size"],
        "value": [raw_n, eff_n]
    })

    df.to_csv(outdir / "inventory_analysis_table.csv", index=False)
    stage_summary.to_csv(outdir / "stage_summary_table.csv", index=False)
    directory_summary.to_csv(outdir / "directory_summary_table.csv", index=False)
    stage_counts.to_csv(outdir / "stage_counts.csv", index=False)
    pac_table.to_csv(outdir / "pac_summary_table.csv", index=False)

    # Figure 1 inventory
    inv = df.groupby(["directory", "stage"]).size().reset_index(name="samples")
    plt.figure(figsize=(8.4, 4.8), dpi=300)
    plt.bar(inv["directory"], inv["samples"])
    plt.xticks(rotation=25, ha="right")
    plt.ylabel("Images processed")
    plt.xlabel("Directory")
    plt.title("Inventory analysis across field-image directories")
    plt.tight_layout()
    plt.savefig(outdir / "fig_inventory_by_directory.png", bbox_inches="tight")
    plt.close()

    # Figure 2 pre/post balance
    plt.figure(figsize=(6.0, 4.2), dpi=300)
    plt.bar(stage_counts["stage"], stage_counts["samples"])
    plt.ylabel("Images processed")
    plt.xlabel("Stage")
    plt.title("Pre- and post-defoliation image balance")
    plt.tight_layout()
    plt.savefig(outdir / "fig_stage_balance.png", bbox_inches="tight")
    plt.close()

    # Figure 3 boll counts
    count_df = stage_summary[stage_summary["stage"].isin(["pre_defoliation", "post_defoliation"])]
    x = np.arange(len(count_df))
    w = 0.35
    plt.figure(figsize=(6.4, 4.4), dpi=300)
    plt.bar(x - w/2, count_df["mean_boll_count_proxy"], width=w, label="Proxy")
    plt.bar(x + w/2, count_df["mean_boll_count_ml"], width=w, label="ML estimate")
    plt.xticks(x, count_df["stage"])
    plt.ylabel("Mean cotton-boll count")
    plt.xlabel("Stage")
    plt.title("Pre/post cotton-boll visibility")
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(outdir / "fig_boll_count_pre_post.png", bbox_inches="tight")
    plt.close()

    # Figure 4 PAC
    plt.figure(figsize=(5.6, 4.2), dpi=300)
    plt.bar(["Raw N", "Effective N"], [raw_n, eff_n])
    plt.ylabel("Sample size")
    plt.title("PAC effective sample size")
    plt.tight_layout()
    plt.savefig(outdir / "fig_pac_effective_sample_size.png", bbox_inches="tight")
    plt.close()

    # Figure 5 PCA
    feat_cols = [
        "mean_r", "mean_g", "mean_b",
        "mean_exg", "std_exg", "mean_ngrdi", "mean_rbr",
        "green_fraction", "bright_fraction",
        "white_region_fraction", "boll_count_ml"
    ]
    qwen_cols = [c for c in df.columns if c.startswith("qwen_")]
    if qwen_cols:
        feat_cols = feat_cols + qwen_cols

    X = df[feat_cols].fillna(0.0).values
    if len(df) >= 3:
        pca = PCA(n_components=2, random_state=42)
        Z = pca.fit_transform(X)
        plot_df = df.copy()
        plot_df["pc1"] = Z[:, 0]
        plot_df["pc2"] = Z[:, 1]

        plt.figure(figsize=(6.6, 5.2), dpi=300)
        for stage in plot_df["stage"].unique():
            sub = plot_df[plot_df["stage"] == stage]
            plt.scatter(sub["pc1"], sub["pc2"], s=18, alpha=0.8, label=stage)
        plt.xlabel("PC1")
        plt.ylabel("PC2")
        plt.title("PCA of ML + Qwen agricultural descriptors")
        plt.legend(frameon=False)
        plt.tight_layout()
        plt.savefig(outdir / "fig_pca_features_qwen.png", bbox_inches="tight")
        plt.close()

    with open(outdir / "paper_notes.txt", "w") as f:
        f.write("Cotton-boll values are proxy/ML estimates, not ground-truth detector counts.\n")
        f.write(f"Raw sample size: {raw_n}\n")
        f.write(f"Effective sample size: {eff_n:.2f}\n")

    print(stage_summary)
    print(pac_table)

root_dir = "/content/drive/MyDrive/TACC EXPERIMENTS"
out_dir = "/content/drive/MyDrive/agrogpt_results_pub"

df = build_dataset(root_dir, limit_per_dir=40, use_qwen=True)
df, model, feature_cols = fit_ml_boll_estimator(df)
make_outputs(df, out_dir)
print("Saved outputs to", out_dir)

In [ ]:
%%writefile LLM_colab_paper_fixed.py
import os
import argparse
import logging
from pathlib import Path
from datetime import datetime
import json

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

ImageFile.LOAD_TRUNCATED_IMAGES = True

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [INFO] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

VALID_EXTS = {'.jpg', '.jpeg', '.png', '.tiff', '.bmp', '.webp'}

def is_valid_image_file(f: Path) -> bool:
    return (
        f.is_file()
        and f.suffix.lower() in VALID_EXTS
        and not f.name.startswith("._")
        and not f.name.startswith(".")
    )

class AgroGPTInference:
    def __init__(self, model_id, device="cuda", dtype=torch.float16):
        self.model_id = model_id
        self.device = device if torch.cuda.is_available() else "cpu"
        self.dtype = dtype if self.device == "cuda" else torch.float32
        self.model = None
        self.processor = None

    def load_model(self):
        logger.info("Started rank=0 world_size=1 local_rank=0")
        logger.info(f"Device selected: {self.device}")
        logger.info(f"Loading model: {self.model_id}")
        if self.device == "cuda":
            logger.info(f"GPU: {torch.cuda.get_device_name(0)}")

        self.processor = AutoProcessor.from_pretrained(self.model_id, use_fast=True)
        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            self.model_id,
            torch_dtype=self.dtype,
            device_map="auto" if self.device == "cuda" else None,
            low_cpu_mem_usage=True
        )

        if self.device == "cpu":
            self.model.to(self.device)

        self.model.eval()
        logger.info("Model loaded successfully.")

    def extract_features(self, image_path):
        try:
            image = Image.open(image_path).convert("RGB")

            messages = [{
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": "Analyze this agricultural UAV image."}
                ]
            }]

            text = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            inputs = self.processor(
                text=[text],
                images=[image],
                return_tensors="pt",
                padding=True
            )
            inputs = {k: v.to(self.device) if hasattr(v, "to") else v for k, v in inputs.items()}

            with torch.no_grad():
                outputs = self.model(**inputs, output_hidden_states=True, return_dict=True)

            hidden = outputs.hidden_states[-1]
            embeddings = hidden.mean(dim=1).detach().float().cpu().numpy()[0]

            img_array = np.array(image)
            hist_features = []
            for c in range(3):
                hist, _ = np.histogram(img_array[:, :, c], bins=32, range=(0, 255))
                hist_features.extend(hist)

            return {
                "embeddings": embeddings.astype(np.float32),
                "histogram": np.array(hist_features, dtype=np.float32),
                "image_shape": img_array.shape
            }

        except Exception as e:
            logger.error(f"Feature extraction failed for {image_path}: {e}")
            return None

    def process_directory(self, dir_path, domain_name, limit=40):
        logger.info(f"=== Processing domain: {domain_name} ===")
        logger.info(f"Directory: {dir_path}")

        results = {
            "domain": domain_name,
            "samples": [],
            "embeddings": [],
            "histograms": [],
            "timestamps": []
        }

        data_path = Path(dir_path)
        if not data_path.exists():
            logger.warning(f"Directory not found: {data_path}")
            return results

        image_files = sorted([f for f in data_path.rglob("*") if is_valid_image_file(f)])[:limit]
        logger.info(f"Found {len(image_files)} valid images")

        if not image_files:
            logger.warning(f"No valid images found in {data_path}")
            return results

        for idx, image_file in enumerate(image_files, 1):
            start = datetime.now()
            features = self.extract_features(str(image_file))
            elapsed = (datetime.now() - start).total_seconds()

            if features is not None:
                results["samples"].append(image_file.name)
                results["embeddings"].append(features["embeddings"])
                results["histograms"].append(features["histogram"])
                results["timestamps"].append(elapsed)

            logger.info(f"domain={domain_name} processed {idx}/{len(image_files)} samples")

        return results

def find_image_directories(root_dir):
    logger.info(f"Scanning for image directories in {root_dir}")
    image_dirs = {}
    root = Path(root_dir)

    for directory in root.rglob("*"):
        if directory.is_dir():
            images = [f for f in directory.iterdir() if is_valid_image_file(f)]
            if images:
                image_dirs[directory.name] = str(directory)
                logger.info(f"Found {len(images)} valid images in {directory.name}")

    return image_dirs

def generate_publication_plots(all_results, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    valid_results = [r for r in all_results if len(r["samples"]) > 0]
    if not valid_results:
        logger.warning("No results to plot")
        return

    domain_df = pd.DataFrame({
        "domain": [r["domain"] for r in valid_results],
        "samples": [len(r["samples"]) for r in valid_results],
        "avg_latency": [np.mean(r["timestamps"]) if r["timestamps"] else 0 for r in valid_results]
    })
    domain_df.to_csv(output_dir / "paper_results_table.csv", index=False)

    plt.figure(figsize=(10, 6), dpi=300)
    plt.bar(domain_df["domain"], domain_df["samples"])
    plt.xticks(rotation=25, ha="right")
    plt.xlabel("Agricultural Domain")
    plt.ylabel("Samples Processed")
    plt.title("AgroGPT Domain-Specific Sample Distribution")
    plt.tight_layout()
    plt.savefig(output_dir / "fig1_domain_distribution.png", bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(10, 6), dpi=300)
    plt.bar(domain_df["domain"], domain_df["avg_latency"])
    plt.xticks(rotation=25, ha="right")
    plt.xlabel("Agricultural Domain")
    plt.ylabel("Average Latency (s)")
    plt.title("Inference Latency Across Domains")
    plt.tight_layout()
    plt.savefig(output_dir / "fig2_latency_analysis.png", bbox_inches="tight")
    plt.close()

def main(args):
    inference = AgroGPTInference(
        model_id=args.model_id,
        device="cuda" if torch.cuda.is_available() else "cpu",
        dtype=torch.float16
    )
    inference.load_model()

    image_dirs = find_image_directories(args.data_dir)
    if not image_dirs:
        logger.error(f"No valid image directories found in {args.data_dir}")
        return

    logger.info(f"Found {len(image_dirs)} directories with valid images: {list(image_dirs.keys())}")

    all_results = []
    for domain_name, dir_path in image_dirs.items():
        result = inference.process_directory(
            dir_path=dir_path,
            domain_name=domain_name,
            limit=args.limit_per_stage
        )
        all_results.append(result)

    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    summary = {
        "experiment": "AgroGPT-Colab-GPU",
        "model": args.model_id,
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "timestamp": datetime.now().isoformat(),
        "domains_found": len(all_results),
        "total_samples": int(sum(len(r["samples"]) for r in all_results)),
        "per_domain": [
            {
                "domain": r["domain"],
                "samples": len(r["samples"]),
                "avg_latency": float(np.mean(r["timestamps"])) if r["timestamps"] else 0
            }
            for r in all_results
        ]
    }

    with open(output_dir / "paper_results.json", "w") as f:
        json.dump(summary, f, indent=2)

    generate_publication_plots(all_results, output_dir)
    logger.info(f"Results saved to {output_dir}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="AgroGPT Publication Results")
    parser.add_argument("--model-id", default="Qwen/Qwen2.5-VL-3B-Instruct")
    parser.add_argument("--limit-per-stage", type=int, default=40)
    parser.add_argument("--output-dir", default="/content/drive/MyDrive/agrogpt_results")
    parser.add_argument("--data-dir", default="/content/drive/MyDrive")
    args, _ = parser.parse_known_args()
    main(args)

In [ ]:
!find "/content/drive/MyDrive/TACC EXPERIMENTS" -type f -name "._*" -delete

In [ ]:
!python LLM_colab_paper_fixed.py \
  --model-id Qwen/Qwen2.5-VL-3B-Instruct \
  --limit-per-stage 40 \
  --data-dir "/content/drive/MyDrive/TACC EXPERIMENTS" \
  --output-dir "/content/drive/MyDrive/agrogpt_results"

In [ ]:
from IPython.display import Image, display
from pathlib import Path

results_dir = Path('/content/drive/MyDrive/agrogpt_results')

print("📊 PUBLICATION-GRADE PLOTS\n")

# Display all plots
for plot_file in sorted(results_dir.glob('fig*.png')):
    print(f"\n{'='*60}")
    print(f"📈 {plot_file.name}")
    print('='*60)
    display(Image(str(plot_file)))

In [ ]:
import os
import json

summary_path = '/content/drive/MyDrive/agrogpt_results/results_summary.txt'
json_path = '/content/drive/MyDrive/agrogpt_results/paper_results.json'

if os.path.exists(summary_path):
    with open(summary_path, 'r') as f:
        print(f.read())
else:
    print(f"File not found: {summary_path}")

if os.path.exists(json_path):
    with open(json_path, 'r') as f:
        results = json.load(f)
        print("\n📊 JSON Results:", json.dumps(results, indent=2))
else:
    print(f"\nFile not found: {json_path}.\nPlease ensure the previous cell (LLM_colab_paper_fixed.py) has finished successfully.")
